In [1]:
# =========================
# DAG Validation (for our saved graphs)
# =========================
import os
import networkx as nx
from typing import List, Tuple, Optional, Any


def _load_graph(path: str) -> nx.DiGraph:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".gexf":
        G = nx.read_gexf(path)
    elif ext in [".graphml", ".xml"]:
        G = nx.read_graphml(path)
    else:
        raise ValueError(f"Unsupported graph format: {ext}. Use .gexf or .graphml")
    return G


def _get_edge_abs_attr(
    G: nx.DiGraph,
    u: Any,
    v: Any,
    prefer: Tuple[str, ...] = ("weight", "score")
) -> Optional[float]:
    """edge attribute 중 prefer 순서대로 찾아 |value| 반환. 없으면 None."""
    data = G.get_edge_data(u, v, default={}) or {}
    for k in prefer:
        if k in data:
            try:
                return abs(float(data[k]))
            except Exception:
                return None
    return None


def validate_dag_graph(
    graph_path: str,
    max_cycles: int = 3,
    show_bidirectional: int = 10,
    attr_prefer: Tuple[str, ...] = ("weight", "score"),
    show_cycle_min_edge: bool = True
) -> None:
    """
    - .gexf / .graphml 그래프가 DAG인지 검증
    - cycle 존재 시 예시 출력
    - 양방향 edge 여부 점검(중복 제거)
    - (옵션) cycle마다 가장 작은 |weight|/|score| edge도 출력
    """
    print("=" * 80)
    print(f"[DAG CHECK] Loading graph: {graph_path}")

    G = _load_graph(graph_path)

    # directed 확인
    is_directed = G.is_directed()
    print(f"[INFO] directed={is_directed}, nodes={G.number_of_nodes()}, edges={G.number_of_edges()}")

    if not is_directed:
        print("[WARN] graph is not directed. DAG check requires a directed graph.")
        print("=" * 80)
        return

    # DAG 여부
    is_dag = nx.is_directed_acyclic_graph(G)
    print(f"[CHECK] is_directed_acyclic_graph: {is_dag}")

    # 양방향(edge conflict) 검사 (u<->v 중복 제거)
    bidirectional_pairs: List[Tuple[Any, Any]] = []
    seen = set()
    for u, v in G.edges():
        if G.has_edge(v, u):
            key = tuple(sorted([str(u), str(v)]))
            if key not in seen:
                seen.add(key)
                bidirectional_pairs.append((u, v))

    if bidirectional_pairs:
        print(f"[WARN] bidirectional edges detected (unique_pairs={len(bidirectional_pairs)}):")
        for u, v in bidirectional_pairs[:show_bidirectional]:
            uv = G.get_edge_data(u, v, default={}) or {}
            vu = G.get_edge_data(v, u, default={}) or {}
            print(f"  - {u} <-> {v} | {u}->{v} attrs={uv} | {v}->{u} attrs={vu}")
        if len(bidirectional_pairs) > show_bidirectional:
            print(f"  ... ({len(bidirectional_pairs) - show_bidirectional} more omitted)")
    else:
        print("[OK] no bidirectional edges")

    # cycle 검사
    if not is_dag:
        print("[ERROR] graph contains cycles")
        cycles = list(nx.simple_cycles(G))
        print(f"[INFO] number of cycles found: {len(cycles)}")

        for idx, cyc in enumerate(cycles[:max_cycles]):
            print(f"  cycle[{idx + 1}] length={len(cyc)}:")
            print("   -> " + " -> ".join([str(x) for x in cyc + [cyc[0]]]))

            if show_cycle_min_edge:
                # cycle edge 중 min |attr| 찾기
                min_edge = None
                min_val = None
                for a, b in zip(cyc, cyc[1:] + [cyc[0]]):
                    val = _get_edge_abs_attr(G, a, b, prefer=attr_prefer)
                    if val is None:
                        continue
                    if (min_val is None) or (val < min_val):
                        min_val = val
                        min_edge = (a, b)

                if min_edge is not None:
                    a, b = min_edge
                    data = G.get_edge_data(a, b, default={}) or {}
                    print(f"    [HINT] smallest |{attr_prefer}| edge in this cycle: {a}->{b}, attrs={data}")
                else:
                    print("    [HINT] could not find numeric edge attributes for cycle edges")
        if len(cycles) > max_cycles:
            print(f"  ... ({len(cycles) - max_cycles} more cycles omitted)")
    else:
        print("[OK] graph is a valid DAG")

    print("=" * 80)


In [4]:
validate_dag_graph("./graph_NOTEARS.gexf")
validate_dag_graph("./graph_PC.gexf")
validate_dag_graph("./graph_GES.gexf")
validate_dag_graph("./graph_GOLEM.gexf")

[DAG CHECK] Loading graph: ./graph_NOTEARS.gexf
[INFO] directed=True, nodes=13, edges=17
[CHECK] is_directed_acyclic_graph: True
[OK] no bidirectional edges
[OK] graph is a valid DAG
[DAG CHECK] Loading graph: ./graph_PC.gexf
[INFO] directed=True, nodes=13, edges=24
[CHECK] is_directed_acyclic_graph: True
[OK] no bidirectional edges
[OK] graph is a valid DAG
[DAG CHECK] Loading graph: ./graph_GES.gexf
[INFO] directed=True, nodes=13, edges=35
[CHECK] is_directed_acyclic_graph: True
[OK] no bidirectional edges
[OK] graph is a valid DAG
[DAG CHECK] Loading graph: ./graph_GOLEM.gexf
[INFO] directed=True, nodes=13, edges=26
[CHECK] is_directed_acyclic_graph: True
[OK] no bidirectional edges
[OK] graph is a valid DAG


In [2]:
# =========================
# layouts_all (Ref-style) - FULL SINGLE-CELL SCRIPT (PAPER-READABLE, NOT OVERDONE)
# - One-cell runnable
# - Strict weight pipeline preserved
# - True arrowheads at edge end (target side)
# - Times New Roman
# - Node colors by in/out rule
# - Edge colors by sign(weight): + blue / - red
# - Paper readability (balanced):
#   * moderate node/font sizes (avoid giant overlaps)
#   * edge labels only TOP-K (default 25) to avoid white blobs
#   * lighter strokes + smaller bboxes
#   * tight-crop save
# - CHANGE REQUESTS APPLIED:
#   (1) Node sizes are fully identical (no degree-based sizing)
#   (2) ✅ Title label 제거 (graph 위에 텍스트 stamp 안 함)
#       -> TOP_TITLE_MARGIN / fig.text / pad_inches(title) 관련 모두 제거
# =========================

import os
import re
import shutil
import subprocess
from typing import Dict, List, Tuple, Callable, Optional, Any

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib import rcParams
import matplotlib.patheffects as pe

# =========================
# Settings
# =========================
BASE_DIR = "."
OUT_BASE = os.path.join(BASE_DIR, "layouts_all")

PATHS = {
    "NOTEARS": os.path.join(BASE_DIR, "graph_NOTEARS.gexf"),
    "GOLEM": os.path.join(BASE_DIR, "graph_GOLEM.gexf"),
    "PC": os.path.join(BASE_DIR, "graph_PC.gexf"),
    "GES": os.path.join(BASE_DIR, "graph_GES.gexf"),
}

# Balanced for paper (not extreme)
FIGSIZE = (14, 10)
DPI = 600

EDGE_W_MIN = 2.0
EDGE_W_MAX = 7.0

TOPK_EDGE_LABELS = 100
TOP_EDGES = None

# Edge label placement
EDGE_LABEL_POS = 0.50
EDGE_LABEL_ROTATE = False
EDGE_LABEL_YOFFSET = 0.006

# Moderate scaling
FONT_SCALE = 1.25
NODE_SIZE_MULT = 1.20

# True arrows
EDGE_ARROWS = True
ARROW_STYLE = "-|>"
ARROW_SIZE = 52
ARROW_ALPHA = 0.95

EDGE_LABEL_BBOX = True

# Graphviz direction (for dot)
DOT_RANKDIR = "TB"

# Save padding (title 제거했으니 최소로)
PAD_INCHES_SAVE = 0.02

LAYOUTS = [
    "spring",
    "kamada_kawai",
    "spectral",
    "circular",
    "shell",
    "random",
    "spiral",
    "multipartite_topo",
    "bfs_layers",
    "radial_layers",
    "spring_tight",
    "spring_loose",
    "graphviz_dot",
    "graphviz_neato",
    "graphviz_fdp",
    "graphviz_sfdp",
    "graphviz_circo",
    "graphviz_twopi",
    "planar_or_fallback",
]

GRAPHVIZ_BIN_CANDIDATES = [
    r"C:\Program Files\Graphviz\bin",
    r"C:\Program Files (x86)\Graphviz\bin",
]

# =========================
# STRICT WEIGHT RULE
# =========================
EPS = 1e-12
PROMOTE_ALT_WEIGHT_KEYS = True
DROP_INVALID_EDGES = False

# =========================
# Font (Times New Roman)
# =========================
rcParams["font.family"] = "Times New Roman"
rcParams["pdf.fonttype"] = 42
rcParams["ps.fonttype"] = 42

# =========================
# Graphviz PATH bootstrap
# =========================
def ensure_graphviz_on_path() -> None:
    if shutil.which("dot") is not None:
        return
    for cand in GRAPHVIZ_BIN_CANDIDATES:
        dot_exe = os.path.join(cand, "dot.exe")
        if os.path.exists(dot_exe):
            os.environ["PATH"] = cand + os.pathsep + os.environ.get("PATH", "")
            break

def has_graphviz() -> bool:
    return shutil.which("dot") is not None

# =========================
# Utils
# =========================
def ensure_dir(p: str) -> None:
    os.makedirs(p, exist_ok=True)

def _as_float(x) -> float:
    try:
        return float(x)
    except Exception:
        return 0.0

def normalize_pos(pos: Dict[str, Tuple[float, float]]) -> Dict[str, Tuple[float, float]]:
    if not pos:
        return pos
    xs = np.array([p[0] for p in pos.values()], dtype=float)
    ys = np.array([p[1] for p in pos.values()], dtype=float)
    xmu, ymu = float(xs.mean()), float(ys.mean())
    xsd, ysd = float(xs.std()), float(ys.std())
    if xsd < 1e-12:
        xsd = 1.0
    if ysd < 1e-12:
        ysd = 1.0
    return {str(k): (float((x - xmu) / xsd), float((y - ymu) / ysd)) for k, (x, y) in pos.items()}

def assert_valid_weighted_graph(G: nx.DiGraph, eps: float = EPS) -> None:
    for u, v, d in G.edges(data=True):
        if "weight" not in d:
            raise ValueError(f"Edge {u}->{v} has no weight")
        w = _as_float(d.get("weight", 0.0))
        if abs(w) <= eps:
            raise ValueError(f"Edge {u}->{v} has zero weight: {w}")

def _extract_weight(data: dict) -> Optional[float]:
    if "weight" in data:
        return _as_float(data.get("weight"))
    if not PROMOTE_ALT_WEIGHT_KEYS:
        return None
    if "score" in data:
        return _as_float(data.get("score"))
    if "w" in data:
        return _as_float(data.get("w"))
    return None

def load_gexf_as_digraph_strict(path: str) -> nx.DiGraph:
    G0 = nx.read_gexf(path)
    G = nx.DiGraph()

    for n, data in G0.nodes(data=True):
        G.add_node(str(n), **data)

    best: Dict[Tuple[str, str], float] = {}

    def consider(u, v, data):
        u2, v2 = str(u), str(v)
        if u2 == v2:
            return
        nd = dict(data) if data is not None else {}
        w = _extract_weight(nd)

        if w is None:
            msg = f"[INVALID] Edge {u2}->{v2} missing weight (weight/score/w not found)."
            if DROP_INVALID_EDGES:
                print(msg + " -> dropped")
                return
            raise ValueError(msg)

        w = float(w)
        if abs(w) <= EPS:
            msg = f"[INVALID] Edge {u2}->{v2} zero weight (<=EPS): {w}"
            if DROP_INVALID_EDGES:
                print(msg + " -> dropped")
                return
            raise ValueError(msg)

        key = (u2, v2)
        if key not in best or abs(w) > abs(best[key]):
            best[key] = w

    if isinstance(G0, nx.MultiDiGraph):
        for u, v, k, data in G0.edges(keys=True, data=True):
            consider(u, v, data)
    else:
        for u, v, data in G0.edges(data=True):
            consider(u, v, data)

    for (u, v), w in best.items():
        G.add_edge(u, v, weight=float(w))

    assert_valid_weighted_graph(G, eps=EPS)
    return G

def filter_edges_strict(G: nx.DiGraph) -> nx.DiGraph:
    H = nx.DiGraph()
    H.add_nodes_from(G.nodes(data=True))

    tmp = []
    for u, v, d in G.edges(data=True):
        w = _as_float(d.get("weight", 0.0))
        if abs(w) <= EPS:
            msg = f"[INVALID] Edge {u}->{v} zero/missing weight at filter: {w}"
            if DROP_INVALID_EDGES:
                print(msg + " -> dropped")
                continue
            raise ValueError(msg)
        tmp.append((u, v, w))

    if TOP_EDGES is not None and len(tmp) > TOP_EDGES:
        tmp = sorted(tmp, key=lambda x: abs(x[2]), reverse=True)[:TOP_EDGES]

    for u, v, w in tmp:
        H.add_edge(u, v, weight=float(w))
    return H

def auto_style_params(G: nx.DiGraph) -> Tuple[int, int, int, int]:
    n = G.number_of_nodes()
    if n <= 15:
        return 2600, 8200, 16, 12
    if n <= 30:
        return 2200, 7200, 14, 11
    if n <= 60:
        return 1800, 6000, 12, 10
    if n <= 120:
        return 1500, 5200, 11, 9
    if n <= 200:
        return 1300, 4600, 10, 9
    return 1100, 3800, 10, 8

# =========================
# Node sizes: FULLY IDENTICAL
# =========================
def compute_node_sizes(G: nx.DiGraph, node_min: int, node_max: int) -> List[float]:
    base = (node_min + node_max) / 2
    return [base for _ in G.nodes()]

def compute_edge_widths(values: List[float]) -> List[float]:
    if not values:
        return []
    absvals = np.array([abs(v) for v in values], dtype=float)
    vmin, vmax = float(absvals.min()), float(absvals.max())
    if vmax - vmin < 1e-12:
        return [2.5 for _ in values]
    out = []
    for a in absvals:
        t = (a - vmin) / (vmax - vmin)
        t = float(np.clip(t, 0.0, 1.0))
        t = np.sqrt(t)
        out.append(EDGE_W_MIN + t * (EDGE_W_MAX - EDGE_W_MIN))
    return out

def connected_components_positions(UG: nx.Graph, layout_fn: Callable[[nx.Graph], Dict]) -> Dict[str, Tuple[float, float]]:
    comps = list(nx.connected_components(UG))
    if not comps:
        return {}
    all_pos: Dict[str, Tuple[float, float]] = {}
    x_offset = 0.0
    for comp in sorted(comps, key=len, reverse=True):
        sub = UG.subgraph(comp).copy()
        pos_c = layout_fn(sub)

        xs = [pos_c[n][0] for n in sub.nodes()] if sub.number_of_nodes() else [0.0]
        minx, maxx = float(min(xs)), float(max(xs))
        width = (maxx - minx) if (maxx - minx) > 1e-9 else 1.0

        for n, (x, y) in pos_c.items():
            all_pos[str(n)] = (float(x + x_offset), float(y))
        x_offset += width * 2.2
    return all_pos

def topo_levels(G: nx.DiGraph) -> Dict[str, int]:
    try:
        order = list(nx.topological_sort(G))
    except Exception:
        return {n: int(G.in_degree(n)) for n in G.nodes()}
    level = {n: 0 for n in G.nodes()}
    for v in order:
        preds = list(G.predecessors(v))
        if preds:
            level[v] = 1 + max(level[p] for p in preds)
    return level

def bfs_layers(G: nx.DiGraph) -> Dict[str, int]:
    roots = [n for n in G.nodes() if G.in_degree(n) == 0]
    if not roots:
        return topo_levels(G)
    UG = G.to_undirected()
    dist = {r: 0 for r in roots}
    from collections import deque
    q = deque(roots)
    while q:
        u = q.popleft()
        for v in UG.neighbors(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return {n: int(dist.get(n, 0)) for n in G.nodes()}

def build_abs_distance_attr(G: nx.Graph, out_attr: str = "dist", min_dist: float = 1e-6) -> None:
    for u, v, d in G.edges(data=True):
        w = _as_float(d.get("weight", 0.0))
        d[out_attr] = float(max(abs(w), min_dist))

# =========================
# Graphviz CLI layout (-Tplain)
# =========================
_PLAIN_NODE_RE = re.compile(
    r'^node\s+(?P<name>"[^"]+"|\S+)\s+(?P<x>-?\d+(\.\d+)?)\s+(?P<y>-?\d+(\.\d+)?)\s+',
    re.M
)

def _unquote(s: str) -> str:
    s = s.strip()
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        return s[1:-1]
    return s

def graphviz_cli_layout(G: nx.DiGraph, prog: str) -> Dict[str, Tuple[float, float]]:
    ensure_graphviz_on_path()
    if not has_graphviz():
        raise RuntimeError("dot not found in PATH for this python process.")

    lines = []
    lines.append("digraph G {")
    if prog == "dot":
        lines.append(f'graph [rankdir="{DOT_RANKDIR}", nodesep="0.30", ranksep="0.45"];')
    else:
        lines.append('graph [overlap="prism"];')
    lines.append('node [shape="ellipse"];')

    for n in G.nodes():
        lines.append(f'"{n}";')
    for u, v in G.edges():
        lines.append(f'"{u}" -> "{v}";')

    lines.append("}")
    dot_in = "\n".join(lines).encode("utf-8")

    cmd = [prog, "-Tplain"]
    p = subprocess.run(cmd, input=dot_in, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if p.returncode != 0:
        raise RuntimeError(f"Graphviz failed: {p.stderr.decode('utf-8', errors='ignore')[:400]}")

    out = p.stdout.decode("utf-8", errors="ignore")

    pos_raw: Dict[str, Tuple[float, float]] = {}
    for m in _PLAIN_NODE_RE.finditer(out):
        name = _unquote(m.group("name"))
        x = float(m.group("x"))
        y = float(m.group("y"))
        pos_raw[name] = (x, y)

    pos: Dict[str, Tuple[float, float]] = {}
    for n in G.nodes():
        ns = str(n)
        if ns in pos_raw:
            pos[ns] = pos_raw[ns]

    if not pos:
        head = "\n".join(out.splitlines()[:40])
        raise RuntimeError(
            "Graphviz succeeded but no node positions were parsed from -Tplain output.\n"
            f"plain head:\n{head}"
        )

    return normalize_pos(pos)

# =========================
# Layout switch
# =========================
def safe_layout(G: nx.DiGraph, layout_name: str) -> Dict[str, Tuple[float, float]]:
    UG = G.to_undirected()

    def spring_default(H): return nx.spring_layout(H, seed=42, weight=None)
    def spring_tight(H): return nx.spring_layout(H, seed=42, weight=None, k=0.10, iterations=300)
    def spring_loose(H): return nx.spring_layout(H, seed=42, weight=None, k=0.35, iterations=200)

    def kk_with_abs_distance(H):
        H2 = H.copy()
        build_abs_distance_attr(H2, out_attr="dist", min_dist=1e-6)
        return nx.kamada_kawai_layout(H2, weight="dist")

    def spectral(H): return nx.spectral_layout(H, weight=None)
    def circular(H): return nx.circular_layout(H)
    def shell(H): return nx.shell_layout(H)
    def random(H): return nx.random_layout(H, seed=42)
    def spiral(H): return nx.spiral_layout(H)

    def multipartite_topo(_):
        levels = topo_levels(G)
        for n, lv in levels.items():
            G.nodes[n]["subset"] = int(lv)
        return nx.multipartite_layout(G, subset_key="subset")

    def bfs_layers_layout(_):
        levels = bfs_layers(G)
        for n, lv in levels.items():
            G.nodes[n]["subset"] = int(lv)
        return nx.multipartite_layout(G, subset_key="subset")

    def radial_layers_layout(_):
        levels = topo_levels(G)
        max_lv = max(levels.values()) if levels else 0
        shells = []
        for lv in range(max_lv + 1):
            shells.append([n for n in G.nodes() if levels.get(n, 0) == lv])
        shells = [s for s in shells if s]
        return nx.shell_layout(UG, nlist=shells)

    def planar_or_fallback(H):
        try:
            pos = nx.planar_layout(H)
            return {str(k): (float(v[0]), float(v[1])) for k, v in pos.items()}
        except Exception:
            return spring_default(H)

    if layout_name == "spring":
        return connected_components_positions(UG, spring_default)
    if layout_name == "spring_tight":
        return connected_components_positions(UG, spring_tight)
    if layout_name == "spring_loose":
        return connected_components_positions(UG, spring_loose)
    if layout_name == "kamada_kawai":
        return connected_components_positions(UG, kk_with_abs_distance)
    if layout_name == "spectral":
        return connected_components_positions(UG, spectral)
    if layout_name == "circular":
        return connected_components_positions(UG, circular)
    if layout_name == "shell":
        return connected_components_positions(UG, shell)
    if layout_name == "random":
        return connected_components_positions(UG, random)
    if layout_name == "spiral":
        return connected_components_positions(UG, spiral)
    if layout_name == "multipartite_topo":
        return multipartite_topo(UG)
    if layout_name == "bfs_layers":
        return bfs_layers_layout(UG)
    if layout_name == "radial_layers":
        return radial_layers_layout(UG)
    if layout_name == "planar_or_fallback":
        return connected_components_positions(UG, planar_or_fallback)

    if layout_name.startswith("graphviz_"):
        prog = layout_name.replace("graphviz_", "")
        if prog == "dot" and not nx.is_directed_acyclic_graph(G):
            prog = "sfdp"
        try:
            return graphviz_cli_layout(G, prog=prog)
        except Exception as e:
            print(f"[GRAPHVIZ FAIL] layout={layout_name} reason={repr(e)} -> fallback to spring")
            return connected_components_positions(UG, spring_default)

    return connected_components_positions(UG, spring_default)

# =========================
# Node coloring rule
# =========================
def node_colors_by_inout(G: nx.DiGraph) -> List[str]:
    colors = []
    for n in G.nodes():
        indeg = G.in_degree(n)
        outdeg = G.out_degree(n)
        if indeg == 0 and outdeg > 0:
            colors.append("red")
        elif indeg > 0 and outdeg > 0:
            colors.append("yellow")
        elif indeg > 0 and outdeg == 0:
            colors.append("green")
        else:
            colors.append("lightgray")
    return colors

# =========================
# Drawing
# =========================
def draw_graph(alg: str, G: nx.DiGraph, layout_name: str, out_dir: str) -> None:
    ensure_dir(out_dir)

    H = filter_edges_strict(G)
    edges = list(H.edges())
    if len(edges) == 0:
        print(f"[SKIP] {alg} | {layout_name}: no edges after filtering")
        return

    pos = safe_layout(H, layout_name)

    # fill missing positions
    if len(pos) < H.number_of_nodes():
        missing = [n for n in H.nodes() if n not in pos]
        fb = nx.spring_layout(H.to_undirected(), seed=42, weight=None)
        for n in missing:
            pos[n] = (float(fb[n][0]), float(fb[n][1]))

    node_min, node_max, font_node, font_edge = auto_style_params(H)
    font_node = int(round(font_node * FONT_SCALE))
    font_edge = int(round(font_edge * FONT_SCALE))

    values = [_as_float(H[u][v].get("weight", 0.0)) for u, v in edges]
    for (u, v), w in zip(edges, values):
        if abs(w) <= EPS:
            raise ValueError(f"[INVALID] {alg} edge {u}->{v} has zero weight at draw: {w}")

    edge_colors = ["tab:blue" if w > 0 else "tab:red" for w in values]
    widths = compute_edge_widths(values)

    # Edge labels: ONLY TOP-K by |weight|
    edge_labels = {}
    if TOPK_EDGE_LABELS and TOPK_EDGE_LABELS > 0:
        order = np.argsort([abs(v) for v in values])[::-1]
        for idx in order[: min(TOPK_EDGE_LABELS, len(order))]:
            u, v = edges[idx]
            edge_labels[(u, v)] = f"{values[idx]:.2f}"

    node_sizes = compute_node_sizes(H, node_min=node_min, node_max=node_max)
    node_sizes = [s * NODE_SIZE_MULT for s in node_sizes]
    node_colors = node_colors_by_inout(H)

    out_png = os.path.join(out_dir, f"{alg}__{layout_name}.png")

    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    # --- arrow margins (node-size aware) ---
    max_node = float(max(node_sizes)) if node_sizes else 2000.0
    node_r = np.sqrt(max_node)
    MS = int(max(10, node_r * 0.10))
    MT = int(max(18, node_r * 0.16))

    # --- Edges first ---
    edge_kwargs = dict(
        G=H,
        pos=pos,
        edgelist=edges,
        edge_color=edge_colors,
        width=widths if widths else 2.2,
        arrows=True,
        arrowstyle=ARROW_STYLE,
        arrowsize=ARROW_SIZE,
        alpha=ARROW_ALPHA,
        connectionstyle="arc3,rad=0.04",
        ax=ax
    )
    try:
        nx.draw_networkx_edges(
            **edge_kwargs,
            min_source_margin=MS,
            min_target_margin=MT
        )
    except TypeError:
        nx.draw_networkx_edges(**edge_kwargs)

    # --- Nodes ---
    nx.draw_networkx_nodes(
        H, pos,
        node_size=node_sizes,
        node_color=node_colors,
        edgecolors="black",
        linewidths=1.8,
        ax=ax
    )

    # --- Node labels (light stroke only) ---
    texts = nx.draw_networkx_labels(
        H, pos,
        font_size=font_node,
        font_family="Times New Roman",
        ax=ax
    )
    for t in texts.values():
        t.set_path_effects([pe.Stroke(linewidth=2.0, foreground="white"), pe.Normal()])
        t.set_zorder(6)

    # --- Edge labels (bbox small + semi-transparent) ---
    if edge_labels:
        et = nx.draw_networkx_edge_labels(
            H, pos,
            edge_labels=edge_labels,
            font_size=font_edge,
            font_family="Times New Roman",
            rotate=EDGE_LABEL_ROTATE,
            label_pos=EDGE_LABEL_POS,
            bbox=(dict(facecolor="white", edgecolor="none", alpha=0.72, pad=0.20) if EDGE_LABEL_BBOX else None),
            ax=ax
        )
        for t in et.values():
            x, y = t.get_position()
            t.set_position((x, y + EDGE_LABEL_YOFFSET))
            t.set_zorder(7)

    ax.set_axis_off()
    fig.savefig(out_png, bbox_inches="tight", pad_inches=PAD_INCHES_SAVE)
    plt.close(fig)

    print(f"[DONE] {alg} | {layout_name} -> {out_png}")

def log_edge_counts(graphs: Dict[str, nx.DiGraph]) -> None:
    for alg, G in graphs.items():
        print(f"{alg}: nodes={G.number_of_nodes()} edges={G.number_of_edges()}")

def main():
    ensure_dir(OUT_BASE)
    ensure_graphviz_on_path()

    if not has_graphviz():
        print("[WARN] Graphviz 'dot' not found in PATH for this Python process.")
        print("       graphviz_* layouts will fallback to spring.")
    else:
        print("[INFO] dot =", shutil.which("dot"))

    graphs: Dict[str, nx.DiGraph] = {}
    for alg, p in PATHS.items():
        if not os.path.exists(p):
            print(f"[SKIP] Missing: {p}")
            continue
        graphs[alg] = load_gexf_as_digraph_strict(p)

    if not graphs:
        raise FileNotFoundError("No graphs to draw. Check BASE_DIR/PATHS.")

    log_edge_counts(graphs)

    for alg, G in graphs.items():
        assert_valid_weighted_graph(G, eps=EPS)

    for alg, G in graphs.items():
        out_dir = os.path.join(OUT_BASE, alg)
        for layout_name in LAYOUTS:
            draw_graph(alg=alg, G=G, layout_name=layout_name, out_dir=out_dir)

    print(f"[ALL DONE] outputs -> {OUT_BASE}")

# Run
main()


[INFO] dot = C:\Program Files (x86)\Graphviz\bin\dot.EXE
NOTEARS: nodes=13 edges=17
GOLEM: nodes=13 edges=26
PC: nodes=13 edges=24
GES: nodes=13 edges=35
[DONE] NOTEARS | spring -> .\layouts_all\NOTEARS\NOTEARS__spring.png
[DONE] NOTEARS | kamada_kawai -> .\layouts_all\NOTEARS\NOTEARS__kamada_kawai.png
[DONE] NOTEARS | spectral -> .\layouts_all\NOTEARS\NOTEARS__spectral.png
[DONE] NOTEARS | circular -> .\layouts_all\NOTEARS\NOTEARS__circular.png
[DONE] NOTEARS | shell -> .\layouts_all\NOTEARS\NOTEARS__shell.png
[DONE] NOTEARS | random -> .\layouts_all\NOTEARS\NOTEARS__random.png
[DONE] NOTEARS | spiral -> .\layouts_all\NOTEARS\NOTEARS__spiral.png
[DONE] NOTEARS | multipartite_topo -> .\layouts_all\NOTEARS\NOTEARS__multipartite_topo.png
[DONE] NOTEARS | bfs_layers -> .\layouts_all\NOTEARS\NOTEARS__bfs_layers.png
[DONE] NOTEARS | radial_layers -> .\layouts_all\NOTEARS\NOTEARS__radial_layers.png
[DONE] NOTEARS | spring_tight -> .\layouts_all\NOTEARS\NOTEARS__spring_tight.png
[DONE] NOTEAR